In [18]:
def _font_ch():
  # 設定中文字體 (Colab 範例)
  # 安裝中文字體
  !pip install fonttools -q
  !apt-get update -qq > /dev/null
  !apt-get install -y fonts-wqy-zenhei > /dev/null

  import matplotlib.pyplot as plt
  import matplotlib.font_manager as fm
  import os

  font_path = '/usr/share/fonts/truetype/wqy/wqy-zenhei.ttc'
  chinese_font_prop = None

  if os.path.exists(font_path):
    # Add font to font manager
    fm.fontManager.addfont(font_path)
    # Set default font properties for matplotlib
    plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'DejaVu Sans', 'Arial Unicode MS'] # Prioritize WenQuanYi Zen Hei
    plt.rcParams['font.family'] = 'sans-serif' # Set the font family
    plt.rcParams['axes.unicode_minus'] = False # 修復負號顯示問題
    print("中文字體 (WenQuanYi Zen Hei) 設定成功。")
    chinese_font_prop = fm.FontProperties(fname=font_path)
  else:
    print(f"警告: 找不到字體檔案 {font_path}。中文字體可能無法正常顯示。")

  print("字體設定完成，請重新執行 Cell 3 以便正確顯示圖表中的中文字體。")
  return chinese_font_prop

# Execute the setup and make the font property globally accessible
global CHINESE_FONT_PROP
CHINESE_FONT_PROP = _font_ch() # 設定中文時，前面#號拿掉

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
中文字體 (WenQuanYi Zen Hei) 設定成功。
字體設定完成，請重新執行 Cell 3 以便正確顯示圖表中的中文字體。


In [19]:

def _inV_():
 !pip install yfinance==0.2.54 # 0.2.51 is err
 #!pip install matplotlib==3.8.1 # 3.10 is err

def _checkV_():
 print()
 import matplotlib ; print("matplotlib",matplotlib.__version__)
 import yfinance ; print("yfinance",yfinance.__version__);print()

def TZ_():
 import datetime
 tzone = datetime.timezone(datetime.timedelta(hours=8))
 now = datetime.datetime.now(tz=tzone); print(now)
# 程式執行 時間計算，A段放最前 Cell ,B段放最後 Cell
# A段
import time
start_time = time.time()


#_inV_()

#_checkV_();TZ_()

In [20]:
# ============================================================
# LSTM 股票交易模型 - 日K預測 (Colab)
# 每支股票獨立訓練，產生各自進出場訊號
# 最多持倉 1，最少持倉 0，不做空
# ============================================================

# ============================================================
# Cell 1: 安裝與設定（先執行此 cell）
# ============================================================
# @title Cell 1 - 安裝套件與匯入

!pip install yfinance -q

import os, sys, copy, pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import yfinance as yf
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import MaxNLocator
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'使用設備: {device}')

# ============================================================
# 股票清單 (代碼, 名稱) - 請在此修改
# ============================================================
STOCKS = [
    ('2330.TW', '台積電'),
    ('2454.TW', '聯發科'),
    ('2308.TW', '台達電'),
    ('2317.TW', '鴻海'),
    ('3711.TW', '日月光投控'),
    ('2383.TW', '台光電'),
    ('3037.TW', '欣興'),
    ('2345.TW', '智邦'),
    ('2881.TW', '富邦金'),
    ('2382.TW', '廣達'),
    ('2882.TW', '國泰金'),
    ('3017.TW', '奇鋐'),
    ('2412.TW', '中華電'),
    ('2891.TW', '中信金'),
    ('2303.TW', '聯電'),
    ('2360.TW', '致茂'),
    ('7769.TW', '鴻華先進'),
]

# ============================================================
# 參數設定
# ============================================================
TRAIN_START = '2022-01-06'
TRAIN_END   = '2025-11-30'
TEST_START  = '2025-01-06'
TEST_END    = '2026-08-30'

SEQ_LEN      = 20
BATCH_SIZE   = 64
HIDDEN_SIZE  = 128
NUM_LAYERS   = 2
DROPOUT      = 0.2
LEARNING_RATE = 0.001
EPOCHS      = 50 # 訓練圈數

MODEL_SAVE_DIR = 'saved_models'

# ============================================================
# 特徵工程
# ============================================================
FEATURE_COLS = [
    'returns', 'log_returns', 'high_low', 'close_open',
    'ma_5_ratio', 'ma_10_ratio', 'ma_20_ratio',
    'volume_ratio', 'rsi_14', 'volatility_20', 'close_position'
]

def add_features(df):
    df = df.copy()
    df['returns'] = df['Close'].pct_change()
    df['log_returns'] = np.log(df['Close'] / df['Close'].shift(1))
    df['high_low'] = (df['High'] - df['Low']) / df['Close']
    df['close_open'] = (df['Close'] - df['Open']) / df['Open']
    for ma in [5, 10, 20]:
        df[f'ma_{ma}'] = df['Close'].rolling(ma).mean()
        df[f'ma_{ma}_ratio'] = df['Close'] / df[f'ma_{ma}'] - 1
    df['volume_ma_5'] = df['Volume'].rolling(5).mean()
    df['volume_ratio'] = df['Volume'] / df['volume_ma_5'].replace(0, np.nan)
    delta = df['Close'].diff()
    gain = delta.where(delta > 0, 0).rolling(14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
    rs = gain / loss.replace(0, np.nan)
    df['rsi_14'] = 100 - (100 / (1 + rs))
    df['volatility_20'] = df['returns'].rolling(20).std()
    df['close_position'] = ((df['Close'] - df['Low'].rolling(20).min()) /
                            (df['High'].rolling(20).max() - df['Low'].rolling(20).min() + 1e-8))
    return df

def create_labels(df, forward=5):
    future_ret = df['Close'].shift(-forward) / df['Close'] - 1
    df['label'] = (future_ret > 0).astype(int)
    return df

def prepare_sequences(data, seq_len=SEQ_LEN):
    vals = data[FEATURE_COLS].values.astype(np.float32)
    labs = data['label'].values.astype(np.float32)
    X, y = [], []
    for i in range(len(vals) - seq_len):
        if np.any(np.isnan(vals[i:i+seq_len])) or np.isnan(labs[i+seq_len]):
            continue
        X.append(vals[i:i+seq_len])
        y.append(labs[i+seq_len])
    if len(X) == 0:
        return np.empty((0, seq_len, len(FEATURE_COLS))), np.empty((0,))
    return np.array(X), np.array(y)

# ============================================================
# LSTM 模型
# ============================================================
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS, dropout=DROPOUT):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0)
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])
        return torch.sigmoid(out).squeeze()

# ============================================================
# 下載資料
# ============================================================
def download_data(symbol, start, end):
    df = yf.download(symbol, start=start, end=end, auto_adjust=True, progress=False)
    if df is None or df.empty:
        return None
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    return df

# ============================================================
# 訓練函數
# ============================================================
def train_model(model, X_train, y_train, X_val=None, y_val=None, epochs=EPOCHS):
    train_dataset = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

    best_loss = float('inf')
    best_state = None

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            optimizer.zero_grad()
            outputs = model(Xb)
            loss = criterion(outputs, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item()

        if X_val is not None:
            model.eval()
            with torch.no_grad():
                Xv = torch.tensor(X_val).to(device)
                yv = torch.tensor(y_val).to(device)
                val_out = model(Xv)
                val_loss = criterion(val_out, yv).item()
            scheduler.step(val_loss)
            if val_loss < best_loss:
                best_loss = val_loss
                best_state = copy.deepcopy(model.state_dict())

        if (epoch + 1) % 10 == 0:
            status = f'Epoch {epoch+1}/{epochs} Train Loss: {train_loss/len(train_loader):.4f}'
            if X_val is not None:
                status += f' Val Loss: {val_loss:.4f}'
            print(f'    {status}')

    if best_state is not None:
        model.load_state_dict(best_state)
    return model

# ============================================================
# 測試函數
# ============================================================
def backtest(model, df, seq_len=SEQ_LEN, threshold=0.5):
    model.eval()
    df = df.reset_index(drop=True)
    signals = np.full(len(df), np.nan)
    data_vals = df[FEATURE_COLS].values.astype(np.float32)

    for i in range(seq_len, len(df)):
        seq = data_vals[i-seq_len:i]
        if np.any(np.isnan(seq)):
            continue
        with torch.no_grad():
            inp = torch.tensor(seq[None, :, :]).to(device)
            prob = model(inp).item()
        signals[i] = prob

    df['signal_prob'] = signals
    df['signal'] = 0
    df.loc[df['signal_prob'] > threshold, 'signal'] = 1
    return df

def calc_performance(df):
    if df.empty or 'signal' not in df.columns:
        return {}
    df = df.copy()
    df['daily_ret'] = df['Close'].pct_change()
    df['strategy_ret'] = df['signal'].shift(1) * df['daily_ret']
    df['cum_market'] = (1 + df['daily_ret']).cumprod()
    df['cum_strategy'] = (1 + df['strategy_ret']).cumprod()

    total_mkt = df['cum_market'].iloc[-1] - 1
    total_strat = df['cum_strategy'].iloc[-1] - 1
    days = len(df)
    years = days / 252
    ann_ret_strat = (df['cum_strategy'].iloc[-1]) ** (1 / years) - 1 if years > 0 else 0
    ann_ret_mkt = (df['cum_market'].iloc[-1]) ** (1 / years) - 1 if years > 0 else 0
    sharpe = np.sqrt(252) * df['strategy_ret'].mean() / df['strategy_ret'].std() if df['strategy_ret'].std() > 0 else 0
    win_rate = (df['strategy_ret'] > 0).sum() / (df['strategy_ret'] != 0).sum() if (df['strategy_ret'] != 0).sum() > 0 else 0

    return {
        'total_return_strategy': total_strat,
        'total_return_buyhold': total_mkt,
        'annual_return_strategy': ann_ret_strat,
        'annual_return_buyhold': ann_ret_mkt,
        'sharpe_ratio': sharpe,
        'win_rate': win_rate,
        'trade_days': (df['strategy_ret'] != 0).sum(),
    }

# ============================================================
# 繪圖函數
# ============================================================
def plot_results(df, symbol, name, perf):
    df = df.reset_index(drop=True)
    fig, axes = plt.subplots(3, 1, figsize=(14, 10), gridspec_kw={'height_ratios': [3, 1, 1]})
    fig.suptitle(f'{name} ({symbol}) - LSTM 交易訊號', fontsize=14, fontweight='bold')

    ax1 = axes[0]
    ax1.plot(df.index, df['Close'], label='收盤價', color='gray', alpha=0.6, linewidth=1)
    buy_signals = df[df['signal'] == 1]
    sell_signals = df[df['signal'] == 0]
    ax1.scatter(buy_signals.index, buy_signals['Close'], marker='^', color='red', s=80,
                alpha=0.8, label='買進訊號', zorder=5)
    ax1.scatter(sell_signals.index, sell_signals['Close'], marker='X', color='blue', s=80,
                alpha=0.8, label='賣出訊號', zorder=5)
    ax1.set_ylabel('價格')
    ax1.legend(loc='upper left')
    ax1.grid(True, alpha=0.3)

    ax2 = axes[1]
    ax2.plot(df.index, df['signal_prob'], label='買進機率', color='purple', linewidth=1)
    ax2.axhline(0.5, color='gray', linestyle='--', alpha=0.5)
    ax2.fill_between(df.index, 0, df['signal_prob'], where=df['signal_prob'] > 0.5,
                     color='red', alpha=0.15, label='買進區域')
    ax2.set_ylabel('機率')
    ax2.set_ylim(-0.05, 1.05)
    ax2.legend(loc='upper left')
    ax2.grid(True, alpha=0.3)

    ax3 = axes[2]
    ax3.plot(df.index, df['cum_strategy'], label=f'策略 ({perf["total_return_strategy"]*100:.1f}%)', color='red', linewidth=1.5)
    ax3.plot(df.index, df['cum_market'], label=f'買進持有 ({perf["total_return_buyhold"]*100:.1f}%)', color='gray', alpha=0.6, linewidth=1)
    ax3.set_ylabel('累積報酬')
    ax3.legend(loc='upper left')
    ax3.grid(True, alpha=0.3)

    plt.tight_layout()
    return fig

def save_table_image(today_df, perf_df, save_path):
    fig, axes = plt.subplots(2, 1, figsize=(12, max(6, len(today_df) * 0.4 + len(perf_df) * 0.4)))
    fig.patch.set_facecolor('white')

    ax1 = axes[0]
    ax1.axis('off')
    ax1.set_title('今日交易訊號總表', fontsize=14, fontweight='bold', pad=10)
    tbl1 = ax1.table(cellText=today_df.values,
                     colLabels=today_df.columns,
                     cellLoc='center', loc='center',
                     colColours=['#1a5276'] * len(today_df.columns))
    tbl1.auto_set_font_size(False)
    tbl1.set_fontsize(10)
    tbl1.scale(1, 1.5)
    for (row, col), cell in tbl1.get_celld().items():
        if row == 0:
            cell.set_text_props(color='white', fontweight='bold')
            cell.set_facecolor('#2c3e50')
        else:
            cell.set_facecolor('#ecf0f1' if row % 2 == 0 else 'white')

    ax2 = axes[1]
    ax2.axis('off')
    ax2.set_title('測試期間績效總表', fontsize=14, fontweight='bold', pad=10)
    tbl2 = ax2.table(cellText=perf_df.values,
                     colLabels=perf_df.columns,
                     cellLoc='center', loc='center',
                     colColours=['#1a5276'] * len(perf_df.columns))
    tbl2.auto_set_font_size(False)
    tbl2.set_fontsize(9)
    tbl2.scale(1, 1.5)
    for (row, col), cell in tbl2.get_celld().items():
        if row == 0:
            cell.set_text_props(color='white', fontweight='bold')
            cell.set_facecolor('#2c3e50')
        else:
            cell.set_facecolor('#ecf0f1' if row % 2 == 0 else 'white')

    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f'表格圖片已儲存: {save_path}')

print('\n✅ Cell 1 完成！所有函數與設定已載入。')

使用設備: cuda

✅ Cell 1 完成！所有函數與設定已載入。


In [21]:

# ============================================================
# Cell 2: 訓練模型（訓練期 2022/01/06 ~ 2025/11/30）
# ============================================================
# @title Cell 2 - 訓練模型（執行此 cell 前須先執行 Cell 1）

print('='*60)
print('開始下載訓練資料與訓練模型')
print(f'訓練期: {TRAIN_START} ~ {TRAIN_END}')
print('='*60)

os.makedirs(MODEL_SAVE_DIR, exist_ok=True)
all_models = {}
training_log = []

download_start = pd.Timestamp(TRAIN_START) - pd.Timedelta(days=120)

for sym, name in STOCKS:
    print(f'\n--- {name} ({sym}) ---')

    df = download_data(sym, download_start.strftime('%Y-%m-%d'), TRAIN_END)
    if df is None or len(df) < SEQ_LEN + 30:
        print(f'  資料不足，跳過')
        continue
    if len(df.columns) >= 5:
        df.columns = ['Close', 'High', 'Low', 'Open', 'Volume']

    df = add_features(df)
    df = create_labels(df)
    df = df.dropna().reset_index(drop=False)

    train_df = df[df['Date'] <= TRAIN_END].copy()
    if len(train_df) < SEQ_LEN + 10:
        print(f'  訓練資料不足，跳過')
        continue

    X, y = prepare_sequences(train_df)
    if len(X) == 0:
        print(f'  訓練序列為空，跳過')
        continue

    val_split = int(len(X) * 0.8)
    X_val, y_val = X[val_split:], y[val_split:]
    X_train, y_train = X[:val_split], y[:val_split]
    print(f'  訓練樣本: {len(X_train)}, 驗證樣本: {len(X_val)}')

    model = LSTMModel(input_size=len(FEATURE_COLS)).to(device)
    model = train_model(model, X_train, y_train, X_val, y_val)

    save_path = os.path.join(MODEL_SAVE_DIR, f'{sym.replace(".", "_")}.pth')
    torch.save({'model_state': model.state_dict(), 'input_size': len(FEATURE_COLS)}, save_path)
    print(f'  模型已儲存: {save_path}')

    all_models[sym] = model
    training_log.append({'symbol': sym, 'name': name, 'train_samples': len(X_train)})

print(f'\n✅ Cell 2 完成！共訓練 {len(all_models)} 個模型。')


開始下載訓練資料與訓練模型
訓練期: 2022-01-06 ~ 2025-11-30

--- 台積電 (2330.TW) ---
  訓練樣本: 788, 驗證樣本: 198
    Epoch 10/50 Train Loss: 0.6927 Val Loss: 0.6925
    Epoch 20/50 Train Loss: 0.6914 Val Loss: 0.6959
    Epoch 30/50 Train Loss: 0.6878 Val Loss: 0.6951
    Epoch 40/50 Train Loss: 0.6853 Val Loss: 0.6946
    Epoch 50/50 Train Loss: 0.6857 Val Loss: 0.6940
  模型已儲存: saved_models/2330_TW.pth

--- 聯發科 (2454.TW) ---
  訓練樣本: 788, 驗證樣本: 198
    Epoch 10/50 Train Loss: 0.6756 Val Loss: 0.7249
    Epoch 20/50 Train Loss: 0.6458 Val Loss: 0.7468
    Epoch 30/50 Train Loss: 0.6387 Val Loss: 0.7553
    Epoch 40/50 Train Loss: 0.6242 Val Loss: 0.7608
    Epoch 50/50 Train Loss: 0.6281 Val Loss: 0.7606
  模型已儲存: saved_models/2454_TW.pth

--- 台達電 (2308.TW) ---
  訓練樣本: 788, 驗證樣本: 198
    Epoch 10/50 Train Loss: 0.6915 Val Loss: 0.6891
    Epoch 20/50 Train Loss: 0.6871 Val Loss: 0.6885
    Epoch 30/50 Train Loss: 0.6861 Val Loss: 0.6925
    Epoch 40/50 Train Loss: 0.6831 Val Loss: 0.6930
    Epoch 50/50 Train L

In [22]:
# Keep
# -*- coding: utf-8 -*-
"""Untitled8.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1Jtm4nEQ-QbUSVcQ-YTV7vRBS0S439LR3
"""

# ============================================================
# Cell 3: 測試模型與輸出結果（測試期 2025/01/06 ~ 2026/08/30）
# ============================================================
# @title Cell 3 - 測試模型與輸出結果（執行此 cell 前須先執行 Cell 1 & 2）

print('='*60)
print('開始測試模型與產生交易訊號')
print(f'測試期: {TEST_START} ~ {TEST_END}')
print('='*60)

if not os.path.exists(MODEL_SAVE_DIR):
    print('錯誤：找不到模型目錄，請先執行 Cell 2 訓練模型。')
    sys.exit(0)

results = []
today_signals = []

download_start_test = pd.Timestamp(TEST_START) - pd.Timedelta(days=120)

for sym, name in STOCKS:
    print(f'\n--- {name} ({sym}) ---')

    model_path = os.path.join(MODEL_SAVE_DIR, f'{sym.replace(".", "_")}.pth')
    if not os.path.exists(model_path):
        print(f'  找不到模型 {model_path}，跳過')
        continue

    df = download_data(sym, download_start_test.strftime('%Y-%m-%d'), TEST_END)
    if df is None or len(df) < SEQ_LEN + 30:
        print(f'  資料不足，跳過')
        continue
    if len(df.columns) >= 5:
        df.columns = ['Close', 'High', 'Low', 'Open', 'Volume']

    df = add_features(df)
    df = create_labels(df)
    df = df.dropna().reset_index(drop=False)

    test_df = df[(df['Date'] >= TEST_START) & (df['Date'] <= TEST_END)].copy()
    if len(test_df) < SEQ_LEN + 5:
        print(f'  測試資料不足，跳過')
        continue

    checkpoint = torch.load(model_path, map_location=device)
    model = LSTMModel(input_size=checkpoint['input_size']).to(device)
    model.load_state_dict(checkpoint['model_state'])
    model.eval()

    X_test, y_test = prepare_sequences(test_df)
    if len(X_test) > 0:
        with torch.no_grad():
            Xt = torch.tensor(X_test).to(device)
            preds = model(Xt).cpu().numpy()
        acc = ((preds > 0.5) == y_test).mean()
        print(f'  測試準確率: {acc:.2%}')

    result_df = backtest(model, test_df)

    # Add the following lines to ensure result_df has 'cum_strategy' and 'cum_market' for plotting
    result_df['daily_ret'] = result_df['Close'].pct_change()
    result_df['strategy_ret'] = result_df['signal'].shift(1) * result_df['daily_ret']
    result_df['cum_market'] = (1 + result_df['daily_ret']).cumprod()
    result_df['cum_strategy'] = (1 + result_df['strategy_ret']).cumprod()

    perf = calc_performance(result_df)

    if perf.get('total_return_strategy') is not None:
        results.append({
            'symbol': sym, 'name': name,
            **{k: v for k, v in perf.items()}
        })

    fig = plot_results(result_df, sym, name, perf)
    plt.savefig(f'{sym.replace(".", "_")}_test_result.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f'  策略報酬: {perf.get("total_return_strategy", 0)*100:.2f}% | 買進持有: {perf.get("total_return_buyhold", 0)*100:.2f}%')

    last = result_df.iloc[-1]
    today_signals.append({
        'symbol': sym, 'name': name,
        'signal': '買進 ▲' if last['signal'] == 1 else '賣出 ✕',
        'probability': last['signal_prob'],
        'close_price': last['Close'],
    })

print('\n\n' + '='*70)
print('今日交易訊號總表')
print('='*70)
today_df = pd.DataFrame(today_signals)
today_df['probability'] = today_df['probability'].apply(lambda x: f'{x:.1%}' if pd.notna(x) else 'N/A')
today_df['close_price'] = today_df['close_price'].apply(lambda x: f'{x:.2f}' if pd.notna(x) else 'N/A')
today_df.columns = ['代碼', '名稱', '訊號', '買進機率', '收盤價']
print(today_df.to_string(index=False))

print('\n\n' + '='*70)
print('測試期間績效總表')
print('='*70)
if results:
    perf_df = pd.DataFrame(results)
    perf_df['total_return_strategy'] = round(perf_df['total_return_strategy'] * 100,1)
    perf_df['total_return_buyhold'] = round(perf_df['total_return_buyhold'] * 100,1)
    perf_df['annual_return_strategy'] = round(perf_df['annual_return_strategy'] * 100,1)
    perf_df['annual_return_buyhold'] = round(perf_df['annual_return_buyhold'] * 100,1)
    perf_df['sharpe_ratio'] = perf_df['sharpe_ratio'].round(2)
    perf_df['win_rate'] = round(perf_df['win_rate'] * 100,1)
    perf_df.columns = ['代碼', '名稱', '策略總報酬%', '買進持有%', '策略年化%', '買進持有年化%',
                        '夏普值', '勝率%', '交易天數']
    print(perf_df.to_string(index=False))
    avg_strat = perf_df['策略總報酬%'].mean()
    avg_bh = perf_df['買進持有%'].mean()
    print(f'\n平均策略報酬: {avg_strat:.2f}% | 平均買進持有: {avg_bh:.2f}%')
else:
    print('無績效資料')
    perf_df = pd.DataFrame()

def save_table_neat(today_df, perf_df, save_path, font_prop=None):
    fig, axes = plt.subplots(2, 1, figsize=(14, max(7, len(today_df) * 0.45 + len(perf_df) * 0.45)))
    fig.patch.set_facecolor('white')

    header_color = '#143250'
    header_text_color = 'white'
    even_color = '#eaf2f8'
    odd_color = 'white'

    ax1 = axes[0]
    ax1.axis('off')
    ax1.set_title('▌今日交易訊號總表', fontsize=15, fontweight='bold', pad=12, loc='left',
                  color='#143250', fontproperties=font_prop)

    tbl1 = ax1.table(cellText=today_df.values,
                     colLabels=today_df.columns,
                     cellLoc='center',
                     loc='center',
                     colColours=[header_color] * len(today_df.columns))
    tbl1.auto_set_font_size(False)
    tbl1.set_fontsize(10)
    tbl1.scale(1, 1.6)

    for (row, col), cell in tbl1.get_celld().items():
        cell.set_edgecolor('#bdc3c7')
        cell.set_linewidth(0.5)
        if row == 0:
            cell.set_text_props(color=header_text_color, fontweight='bold', fontproperties=font_prop)
            cell.set_facecolor(header_color)
            cell.set_height(cell.get_height() * 1.2)
        else:
            cell.set_facecolor(even_color if row % 2 == 0 else odd_color)
            cell.set_text_props(fontproperties=font_prop)

    ax2 = axes[1]
    ax2.axis('off')
    ax2.set_title('▌測試期間績效總表', fontsize=15, fontweight='bold', pad=12, loc='left',
                  color='#143250', fontproperties=font_prop)

    tbl2 = ax2.table(cellText=perf_df.values,
                     colLabels=perf_df.columns,
                     cellLoc='center',
                     loc='center',
                     colColours=[header_color] * len(perf_df.columns))
    tbl2.auto_set_font_size(False)
    tbl2.set_fontsize(9)
    tbl2.scale(1, 1.6)

    for (row, col), cell in tbl2.get_celld().items():
        cell.set_edgecolor('#bdc3c7')
        cell.set_linewidth(0.5)
        if row == 0:
            cell.set_text_props(color=header_text_color, fontweight='bold', fontproperties=font_prop)
            cell.set_facecolor(header_color)
            cell.set_height(cell.get_height() * 1.2)
        else:
            cell.set_facecolor(even_color if row % 2 == 0 else odd_color)
            cell.set_text_props(fontproperties=font_prop)

    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f'表格圖片已儲存: {save_path}')

today_str = datetime.now().strftime('%Y%m%d')
base_name = f'Bull_Bear_{today_str}'
save_dir = '/content/output_tables' # Changed to a Colab-compatible path
os.makedirs(save_dir, exist_ok=True) # Ensure the directory exists
save_path = os.path.join(save_dir, f'{base_name}.jpg')

if os.path.exists(save_path):
    suffix = 'b'
    while os.path.exists(os.path.join(save_dir, f'{base_name}_{suffix}.jpg')):
        suffix = chr(ord(suffix) + 1)
    save_path = os.path.join(save_dir, f'{base_name}_{suffix}.jpg')

if len(today_signals) > 0:
    today_display = today_df.copy()
    today_display['買進機率'] = today_display['買進機率'].apply(lambda x: f'{x:.1%}' if isinstance(x, (int, float)) and pd.notna(x) else str(x))
    today_display['收盤價'] = today_display['收盤價'].apply(lambda x: f'{x:.2f}' if isinstance(x, (int, float)) and pd.notna(x) else str(x))
else:
    today_display = pd.DataFrame({'代碼': [], '名稱': [], '訊號': [], '買進機率': [], '收盤價': []})

if results:
    perf_display = perf_df.copy()
    for col in ['策略總報酬%', '買進持有%', '策略年化%', '買進持有年化%', '勝率%']:
        if col in perf_display.columns:
            perf_display[col] = perf_display[col].apply(lambda x: f'{x:.1f}%')
    if '夏普值' in perf_display.columns:
        perf_display['夏普值'] = perf_display['夏普值'].apply(lambda x: f'{x:.2f}')
else:
    perf_display = pd.DataFrame({'代碼': [], '名稱': [], '策略總報酬%': [], '買進持有%': [], '策略年化%': [], '買進持有年化%': [], '夏普值': [], '勝率%': [], '交易天數': []})

save_table_neat(today_display, perf_display, save_path, font_prop=globals().get('CHINESE_FONT_PROP'))

print('\n✅ Cell 3 完成！所有測試與輸出已產生。')

開始測試模型與產生交易訊號
測試期: 2025-01-06 ~ 2026-08-30

--- 台積電 (2330.TW) ---
  測試準確率: 61.30%
  策略報酬: 109.03% | 買進持有: 104.38%

--- 聯發科 (2454.TW) ---
  測試準確率: 50.46%
  策略報酬: 190.84% | 買進持有: 203.08%

--- 台達電 (2308.TW) ---
  測試準確率: 65.94%
  策略報酬: 458.39% | 買進持有: 435.04%

--- 鴻海 (2317.TW) ---
  測試準確率: 45.82%
  策略報酬: -4.49% | 買進持有: 47.72%

--- 日月光投控 (3711.TW) ---
  測試準確率: 60.06%
  策略報酬: 237.98% | 買進持有: 236.96%

--- 台光電 (2383.TW) ---
  測試準確率: 61.30%
  策略報酬: 267.85% | 買進持有: 707.23%

--- 欣興 (3037.TW) ---
  測試準確率: 43.65%
  策略報酬: 64.71% | 買進持有: 490.68%

--- 智邦 (2345.TW) ---
  測試準確率: 61.61%
  策略報酬: 233.91% | 買進持有: 209.55%

--- 富邦金 (2881.TW) ---
  測試準確率: 52.01%
  策略報酬: 41.31% | 買進持有: 43.62%

--- 廣達 (2382.TW) ---
  測試準確率: 53.56%
  策略報酬: 54.52% | 買進持有: 35.60%

--- 國泰金 (2882.TW) ---
  測試準確率: 54.49%
  策略報酬: 53.32% | 買進持有: 53.09%

--- 奇鋐 (3017.TW) ---
  測試準確率: 61.61%
  策略報酬: 330.50% | 買進持有: 279.69%

--- 中華電 (2412.TW) ---
  測試準確率: 55.42%
  策略報酬: 7.32% | 買進持有: 21.10%

--- 中信金 (2891.TW) ---
  測試準確率: 58.20%
  策略報酬: 78

In [23]:
# 執行一些程式碼
# B段
end_time = time.time()
execution_time = end_time - start_time
t22=execution_time/60
t2= round(t22, 2)
print("程式執行時間：", execution_time, "秒")
print("程式執行時間：",t2," min")
# https://vocus.cc/article/64931d14fd8978000163c7d3

程式執行時間： 92.70197463035583 秒
程式執行時間： 1.55  min


In [24]:
from IPython.display import Audio, display
import time

def main_program():
    print("程式正在執行中，請稍候...")
    # 模擬耗時的運算
    time.sleep(5)
    print("運算完成！")

# 程式主邏輯
main_program()

# --- 程式結束時的提示音 ---

# 產生一個頻率為 440 Hz (A4) 的 1.0 秒鐘正弦波作為提示音
# 數據使用 numpy 產生 (需要先確保 numpy 已安裝，Colab 預設有)
try:
    import numpy as np

    # 參數設定
    samplerate = 44100
    duration = 0.3 # 聲音持續時間 1 秒
    frequency = 280 # 440 Hz

    t = np.linspace(0., duration, int(samplerate*duration))
    # 產生音訊波形
    note = 0.5 * np.sin(2.*np.pi*frequency*t)

    # 播放音訊
    print("播放提示音...")
    display(Audio(note, rate=samplerate, autoplay=True))

except ImportError:
    print("警告：缺少 numpy 模組，無法產生聲音。")

程式正在執行中，請稍候...
運算完成！
播放提示音...
